In [ ]:
# Dubai Property Listing Attribute Segmentation — Illustrative Rules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
%matplotlib inline

# 1. Load your file
df = pd.read_csv('bayut_selling_properties.csv')   # ← your exact file

print("Original shape:", df.shape)

# 2. Clean numeric columns
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['beds'] = pd.to_numeric(df['beds'], errors='coerce')
df['baths'] = pd.to_numeric(df['baths'], errors='coerce')

# 3. Arabic detection function (an exploratory text feature (not evidence of buyer intent))
def has_arabic(text):
    return bool(re.search('[\u0600-\u06FF]', str(text))) if pd.notna(text) else False

df['has_arabic'] = (df['building_name'].apply(has_arabic) | 
                    df['address'].apply(has_arabic)).astype(int)

# 4. Rule-Based Scoring – Illustrative property attribute scoring
score = 0

# Price tier (higher listing price = higher analyst-assigned score)
score += df['price'].apply(lambda x: 
    50 if x >= 10_000_000 else
    35 if x >= 5_000_000 else
    20 if x >= 2_000_000 else
    10 if x >= 1_000_000 else 5
)

# Selected locations = +30 points (subjective selection)
hot_areas = ['Dubai Marina','Downtown Dubai','Palm Jumeirah','Jumeirah Beach Residence',
             'Business Bay','Dubai Hills Estate','Emirates Hills','Bluewaters','JBR','Arabian Ranches']
df['is_hot_area'] = df['area_name'].str.contains('|'.join(hot_areas), case=False, na=False).astype(int)
score += df['is_hot_area'] * 30

# Size & luxury
score += (df['beds'] >= 4).astype(int) * 20
score += (df['baths'] >= 4).astype(int) * 15

# Ready status receives additional points; speed of sale not established
score += df['completion_status'].str.contains('Ready', case=False, na=False).astype(int) * 25

# Arabic script indicator (heuristic only; not evidence of demand)
score += df['has_arabic'] * 15

# Final columns
df['listing_score'] = score.clip(0, 200)
df['listing_segment'] = pd.cut(df['listing_score'], 
                         bins=[-float('inf'), 50, 90, float('inf')],
                         labels=['Low (<=50)','Medium (51-90)','High (>90)'])

print("Score distribution:")
print(df['listing_segment'].value_counts().sort_index())

# Top 10 highest-scored listings
top10 = df.nlargest(10, 'listing_score')[['building_name','area_name','price','beds','baths',
                                        'completion_status','has_arabic','listing_score','listing_segment']]
print("\nTop 10 highest-scored listings:")
print(top10)          # ← changed from "top" to "top10"

# Save everything
# Row-level scored exports are intentionally not written in the public notebook.
print("\nRow-level exports are disabled in this public notebook.")

# Beautiful chart
plt.figure(figsize=(10,5))
df['listing_segment'].value_counts().sort_index().plot(kind='barh', color=['#7f8c8d','#f39c12','#e74c3c'])
plt.title('Dubai Property Listing Segments (Rule-Based Scoring)')
plt.xlabel('Number of Properties')
plt.tight_layout()
plt.show()